<a href="https://colab.research.google.com/github/malyadri6761/nlp/blob/main/NLP_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

**Term Frequency(TF)**

---

```
  TF(t,d) = (No.of times term t appears in document d) / (Total No.of terms in document d)

```

---

**Pure Python**

---

In [ ]:
# !pip install pypdf
from collections import Counter
from pypdf import PdfReader

path = "/content/CS509_lab_test_2.pdf"
reader = PdfReader(path)

text = ""

for index,pages in enumerate(reader.pages):
  string = pages.extract_text()
  text+=string

words = text.lower().split()
word_count = Counter(words)
total_words = len(words)
tf_scores = {word : count/total_words for word,count in word_count.items()}
print(tf_scores)

import pprint
pprint.pprint(tf_scores)

{'cs509': 0.0003291639236339697, '–': 0.0013166556945358788, 'pgsl': 0.0003291639236339697, 'lab': 0.0003291639236339697, 'test': 0.0013166556945358788, '2': 0.0009874917709019092, 'max.': 0.0003291639236339697, 'marks:': 0.0013166556945358788, '50': 0.0003291639236339697, 'weightage:': 0.0003291639236339697, '20%': 0.0003291639236339697, 'date:': 0.0003291639236339697, '10th': 0.0003291639236339697, 'sept': 0.0003291639236339697, '2026': 0.0003291639236339697, 'duration:': 0.0003291639236339697, '2:30pm': 0.0003291639236339697, 'to': 0.017116524028966424, '5:45pm': 0.0006583278472679394, 'instructions:': 0.0003291639236339697, '1.': 0.0003291639236339697, 'all': 0.0016458196181698486, 'your': 0.014154048716260697, 'code': 0.002304147465437788, 'changes': 0.0006583278472679394, 'should': 0.004279131007241606, 'be': 0.0069124423963133645, 'on': 0.0065832784726793945, 'the': 0.045753785385121794, 'base': 0.0003291639236339697, 'you': 0.007570770243581304, 'have': 0.0009874917709019092, '

---

**Using Scikit-Learn**

---

```
  TF is rarely claculated alone; It is usually paired with Inverse Document Frequency(TF-IDF).
  Scikit-Learn's TfidfVectorizer uses L2 normalization by default to prevent longer documents from drowing out shorter ones.

```

In [ ]:
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer

# Assume each page one document.
def read_pdf(path):
  reader = PdfReader(path)
  text = []
  for index,pages in enumerate(reader.pages):
    string = pages.extract_text()
    text.append(string)
  return text


path = "/content/CS509_lab_test_2.pdf"
text = read_pdf(path)

# Vectorizer (use_idf = False extracts just the TF part)
# norm = None keeps it from adjusting the scores based on document length
vectorizer = TfidfVectorizer(use_idf=False,norm=None)

# Fit and transform the corpus
tf_matrix = vectorizer.fit(text)

feature_names = vectorizer.get_feature_names_out()

dt_tf = pd.DataFrame(tf_matrix.transform(text).toarray(),columns=feature_names)
print(dt_tf)

print(f'Shape : {dt_tf.shape}')



    10  10th   15   20  ...  yourexistingglobal  zero  αab   βc
0  1.0   1.0  1.0  1.0  ...                 0.0   0.0  0.0  0.0
1  0.0   0.0  0.0  0.0  ...                 0.0   0.0  1.0  1.0
2  0.0   0.0  0.0  0.0  ...                 0.0   0.0  0.0  0.0
3  0.0   0.0  0.0  0.0  ...                 0.0   0.0  0.0  0.0
4  0.0   0.0  0.0  0.0  ...                 1.0   0.0  0.0  0.0
5  0.0   0.0  0.0  0.0  ...                 0.0   0.0  0.0  0.0
6  0.0   0.0  0.0  0.0  ...                 0.0   2.0  0.0  0.0

[7 rows x 816 columns]
Shape : (7, 816)


---

**IDF - (Inverse Document Frequency)**

---

```
  Text processing and information retrieval, IDF measures how important or rare a term is acrosss an entire set of documents.

  IDF(t,D) = log ( Total No.of documents in corpus(N))/ ( No.of Documents containing term t+1)

  Note : Adding 1 to the denominator is a common practice called smooth IDF.
  It prevents a "divide by zero" error if a term isn't found in any document).

  Example :    
    Doc 1: "The cat sat on the mat.
    Doc 2: "Dogs hate the cat.
    Doc 3: "A bird flew over the house.

    Case 1: The word "the"
    Total documents (N): 3
    Documents containing "the": 3 (It appears in all three documents).
    
    Smooth IDF Calculation:\[\text{IDF}(\text{"the"})=\log \left(\frac{3}{3+1}\right)=\log (0.75)\approx -0.28\]
  ```

---

**Pure Python**

---

In [ ]:
import math
from collections import Counter

path = "/content/CS509_lab_test_2.pdf"
text = read_pdf(path)

# Tokenize documents into sets of unique words
doc_words = [set(doc.lower().split()) for doc in text]
total_docs = len(doc_words)

# Count how many documents contain each word
doc_frequency = Counter()
for unique_words in doc_words:
  doc_frequency.update(unique_words)


# calculate smooth IDF for each word
idf_score = {}
for word,count in doc_frequency.items():
  idf_score[word] = math.log((1+total_docs)/(1+count))+1

# Print the results sorted by highest importance
print(f'{'Word':<10}| {'Doc count':<10}| {'IDF Score'}')
for word,score in sorted(idf_score.items(),key=lambda x:x[1],reverse=True):
  print(f'{word:<10}| {doc_frequency[word]:<10}| {score}')


Word      | Doc count | IDF Score
alone.    | 1         | 2.386294361119891
70        | 1         | 2.386294361119891
assign    | 1         | 2.386294361119891
4.for     | 1         | 2.386294361119891
1.        | 1         | 2.386294361119891
50        | 1         | 2.386294361119891
compressed| 1         | 2.386294361119891
2,        | 1         | 2.386294361119891
marks:    | 1         | 2.386294361119891
loop.     | 1         | 2.386294361119891
cohesive  | 1         | 2.386294361119891
separately.| 1         | 2.386294361119891
demonstrating| 1         | 2.386294361119891
base      | 1         | 2.386294361119891
basically | 1         | 2.386294361119891
4.        | 1         | 2.386294361119891
learned   | 1         | 2.386294361119891
duration: | 1         | 2.386294361119891
positions | 1         | 2.386294361119891
5:45pm    | 1         | 2.386294361119891
mins      | 1         | 2.386294361119891
updating  | 1         | 2.386294361119891
metric    | 1         | 2.386294361119

---

**IDF - Scikit-learn**

---

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


text = read_pdf(path)

vectorizer = TfidfVectorizer()
vectorizer.fit(text)

# Extracting features
features = vectorizer.get_feature_names_out()
idf_scores = vectorizer.idf_

# Pairing features and idf_scores
df = pd.DataFrame({
    'Word' : features,
    'Scikit IDF Score': idf_scores
})

# Sort by the highest IDF Score (rarest words first)
df = df.sort_values(by='Scikit IDF Score',ascending = False).reset_index(drop=True)
print(df)

         Word  Scikit IDF Score
0        wise          2.386294
1       white          2.386294
2       while          2.386294
3     whether          2.386294
4    whatever          2.386294
..        ...               ...
811   program          1.000000
812        be          1.000000
813       and          1.000000
814       are          1.000000
815        as          1.000000

[816 rows x 2 columns]


---

**Navie Bayes for text - Calculate word Probabilities**

---

```
  P(wi/C) = (Nic + 1)/(Nc+V)

  wi : The specific word you are calculating the probability for.
  C : The class (eg : Spam or Not spam)
  Nic : Total count of how many times word wi appears inside class C.
  Nc : Total count of all words inside class C combined.
  V : The size of the Vocabulary(The No.of unique words across all classes combined).
  

---

**Logistic Regression - Text Classification**

---

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd

corpus = [
    "buy crypto now",      # Spam (1)
    "crypto cash bonus",   # Spam (1)
    "now cash the check",  # Ham  (0)
    "please check the account balance" # Ham (0)
]

labels = [1,1,0,0]


In [ ]:
# Text into Numerical Vectors
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(corpus)

In [ ]:
# Train Logistic Regression
model = LogisticRegression()
model.fit(X_train,labels)

LogisticRegression()

In [ ]:
# Learned coefficient(weight)
vocabulary = vectorizer.get_feature_names_out()
weights  = model.coef_[0]
df = pd.DataFrame({
    'Word' : vocabulary,
    'Weight' : weights
})
print(df)

      Word    Weight
0  account -0.208638
1  balance -0.208638
2    bonus  0.258596
3      buy  0.258596
4     cash -0.049936
5    check -0.517170
6   crypto  0.517191
7      now -0.049936
8   please -0.208638
9      the -0.517170


In [ ]:
# Predict unseen data
test = ["buy cash"]
X_test = vectorizer.transform(test)
prediction = model.predict(X_test)
probabilities = model.predict_proba(X_test)

if prediction ==1 :
  print(f'Prediction : Spam')
else:
  print(f'Prediction : Ham')
probabilites_df = pd.DataFrame({
    'Class' : ['Spam','Ham'],
    'Probability' : probabilities[0]
})
print(probabilites_df)
print(probabilities)

Prediction : Spam
  Class  Probability
0  Spam     0.369107
1   Ham     0.630893
[[0.3691066 0.6308934]]
